### Import

In [1]:
import numpy as np
import pandas as pd
import os
import glob
from scipy.stats import zscore
from nltools.stats import isc

### Configuration of Paths, Variables

In [2]:
DATA_DIR = "../data/02_processed_combined_data"
OUTPUT_DIR = "../data/04_analysis/h1_results/manipulated_audience_for_revcorr/"
if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)

signals = ['Gaze_X', 'Gaze_Y', 'Pupil_Mean', 'Heart_Rate', 'GSR']
N_SAMPLES = 5000
current_dir = os.getcwd()
CONDITIONS_FILE = os.path.abspath(os.path.join(current_dir, "../data/sub_conditions.csv"))

### Set up Helper Functions

In [3]:
def get_subjects_by_condition(conditions_file, target_condition='PC', exclude_prior_exposure=False):
    """
    Reads the conditions file and returns a list of subject IDs for the target condition.
    
    Parameters:
    - exclude_prior_exposure (bool): If True, removes subjects where 'Seen' == 1.
    """
    if not os.path.exists(conditions_file):
        raise FileNotFoundError(f"Conditions file not found: {conditions_file}")
        
    df = pd.read_csv(conditions_file)
    
    # Clean column names
    df.columns = [c.strip() for c in df.columns]
    
    # Check required columns
    required = ['RespondentName', 'SourceStimulus']
    if not set(required).issubset(df.columns):
        raise ValueError(f"Conditions file must contain {required} columns.")
    
    # 1. Filter by Target Condition first
    condition_df = df[df['SourceStimulus'].str.strip() == target_condition].copy()
    
    # 2. Filter by Prior Exposure (New Logic)
    if exclude_prior_exposure:
        if 'Seen' in condition_df.columns:
            # Get initial count for reporting
            n_before = len(condition_df)
            
            # Filter: Keep rows where Seen is NOT 1 (handles 0, NaN, or empty)
            # We convert to numeric to be safe against "1" (str) vs 1 (int)
            condition_df['Seen'] = pd.to_numeric(condition_df['Seen'], errors='coerce').fillna(0)
            condition_df = condition_df[condition_df['Seen'] != 1]
            
            n_excluded = n_before - len(condition_df)
            if n_excluded > 0:
                print(f"[{target_condition}] Excluded {n_excluded} subjects due to prior exposure.")
        else:
            print(f"Warning: 'exclude_prior_exposure' was True, but 'Seen' column was not found in {conditions_file}.")

    # 3. Extract IDs
    subject_ids = condition_df['RespondentName'].apply(parse_subject_id).tolist()
    
    # Remove duplicates and sort
    subject_ids = sorted(list(set(subject_ids)))
    
    return subject_ids

def parse_subject_id(respondent_name):
    """
    Extracts subject ID (e.g., 'sub001') from RespondentName string.
    Handles potential extra spaces or underscores.
    """
    # Split by underscore and take the first part
    parts = respondent_name.split('_')
    sub_id = parts[0].strip()
    return sub_id

def load_filtered_physiological_data(folder_path, subject_list):
    """
    Loads CSV files ONLY if they match an ID in subject_list.
    Checks for required columns and data integrity.
    Returns a dictionary of DataFrames (Time x Subjects).
    """
    
    required_cols = ['Time_sec', 'GSR', 'Heart_Rate', 'Pupil_Mean', 'Gaze_X', 'Gaze_Y']
    signal_types = ['GSR', 'Heart_Rate', 'Pupil_Mean', 'Gaze_X', 'Gaze_Y']
    
    # 1. Structure to hold data: {'GSR': [], 'Heart_Rate': [], ...}
    data_collector = {signal: [] for signal in signal_types}
    
    file_list = glob.glob(os.path.join(folder_path, "*.csv"))
    
    print(f"Scanning {len(file_list)} files against {len(subject_list)} subject IDs...")
    
    loaded_count = 0
    
    for filepath in file_list:
        filename = os.path.basename(filepath)
        
        # --- FILTER 1: CHECK FILENAME AGAINST LIST ---
        # We look for a subject ID that exists inside this filename.
        # e.g., if filename is 'sub003_combined_physio.csv' and 'sub003' is in the list
        matched_subject_id = None
        
        for subj_id in subject_list:
            if subj_id in filename:
                matched_subject_id = subj_id
                break # Found the match, stop checking other IDs
        
        # If this file doesn't match anyone in our list, skip it silently
        if matched_subject_id is None:
            continue

        try:
            # Load data
            df = pd.read_csv(filepath)
            
            # --- FILTER 2: CHECK COLUMN EXISTENCE ---
            if not set(required_cols).issubset(df.columns):
                print(f"Skipping {matched_subject_id} (File: {filename}) - Missing required columns")
                continue
            
            # --- FILTER 3: CHECK FOR EMPTY/NAN DATA ---
            # Check if any required column has NaN values
            if df[required_cols].isnull().any().any():
                print(f"Skipping {matched_subject_id} (File: {filename}) - Contains NaNs/Missing data")
                continue
                
            # --- PROCESSING ---
            # 1. Handle Time Index (Round to 3 decimals to ensure alignment across subjects)
            df['Time_sec'] = df['Time_sec'].round(3)
            df = df.set_index('Time_sec')
            
            # 2. Add each signal to its respective list
            for signal in signal_types:
                s = df[signal]
                s.name = matched_subject_id  # Column name becomes 'sub002', etc.
                data_collector[signal].append(s)
                
            loaded_count += 1
            
        except Exception as e:
            print(f"Error reading {filename}: {e}")

    print(f"Successfully loaded {loaded_count} valid subjects.")

    # --- MERGING ---
    final_dfs = {}
    for signal, series_list in data_collector.items():
        if series_list:
            # Concat aligns on the Index (Time_sec) automatically
            merged_df = pd.concat(series_list, axis=1)
            merged_df.sort_index(inplace=True)
            final_dfs[signal] = merged_df
        else:
            final_dfs[signal] = pd.DataFrame()

    return final_dfs

def get_zscored_split_half(data_df):
    """
    Creates ONE fixed split-half for the timecourse plot.
    Z-scores them so they share a visual scale.
    """
    data = data_df.values
    n_subjects = data.shape[1]
    
    # Use fixed seed for reproducibility of the plot
    np.random.seed(42) 
    indices = np.arange(n_subjects)
    np.random.shuffle(indices)
    mid = n_subjects // 2
    
    g1 = data[:, indices[:mid]].mean(axis=1)
    g2 = data[:, indices[mid:]].mean(axis=1)
    
    # Z-score
    g1_z = zscore(g1, nan_policy='omit')
    g2_z = zscore(g2, nan_policy='omit')
    
    return g1_z, g2_z, np.corrcoef(g1_z, g2_z)[0, 1]

### Precompute Results

In [4]:
# important: This is for the manipulated movie (PCM)!
pc_subjs = get_subjects_by_condition(CONDITIONS_FILE, target_condition='PCM', exclude_prior_exposure=False) 
results = load_filtered_physiological_data(DATA_DIR, pc_subjs)

summary_stats = []

print("Starting Heavy Computations...")

for signal in signals:
    print(f"Processing {signal}...")
    df = results[signal]
    
    # 1. SPLIT-HALF TIMECOURSE (For visual plotting)
    g1, g2, r_split = get_zscored_split_half(df)
    
    # Save Timecourse to CSV
    tc_df = pd.DataFrame({'Time_Sec': df.index, 'Group_1': g1, 'Group_2': g2})
    tc_df.to_csv(os.path.join(OUTPUT_DIR, f"{signal}_split_timecourse.csv"), index=False)

    # 2. PHASE RANDOMIZATION (The Heavy 5000 permutations)
    # Note: nltools isc expects data to be shaped depending on input, usually (time, subjects)
    stats_phase = isc(df, method='phase_randomize', n_samples=N_SAMPLES, return_null=True)
    
    # Extract values
    obs_isc = stats_phase['isc']
    p_val = stats_phase['p']
    null_dist = stats_phase['null_distribution']
    
    # Save the Null Distribution Array (so we can plot histograms later)
    np.save(os.path.join(OUTPUT_DIR, f"{signal}_null_dist.npy"), null_dist)
    
    # 3. SAVE SUMMARY STATS
    summary_stats.append({
        'Signal': signal,
        'Split_Half_r': r_split,
        'Observed_ISC': obs_isc,
        'Phase_Rand_P': p_val
    })

# Save Master Stats Table
stats_df = pd.DataFrame(summary_stats)
stats_df.to_csv(os.path.join(OUTPUT_DIR, "h1_summary_stats.csv"), index=False)
print(f"All computations saved to '{OUTPUT_DIR}' directory.")

Scanning 195 files against 99 subject IDs...
Skipping sub013 (File: sub013_combined_physio.csv) - Contains NaNs/Missing data
Successfully loaded 96 valid subjects.
Starting Heavy Computations...
Processing Gaze_X...
Processing Gaze_Y...
Processing Pupil_Mean...
Processing Heart_Rate...
Processing GSR...
All computations saved to '../data/04_analysis/h1_results/manipulated_audience_for_revcorr/' directory.
